In [ ]:
# ===== simplest_logreg.py =====
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score
)

# ---------------------------
# 1) Load data
# ---------------------------
df_train = pd.read_csv('../data/FEwithMerchants/FE_train_with_merchants.csv')
df_val   = pd.read_csv('../data/FEwithMerchants/FE_validation_with_merchants.csv')
df_test  = pd.read_csv('../data/FEwithMerchants/FE_test_with_merchants.csv')

# (Use these instead if you want the "without merchants" version)
# df_train = pd.read_csv('../data/FEwithoutMerchants/FE_train_without_merchants.csv')
# df_val   = pd.read_csv('../data/FEwithoutMerchants/FE_validation_without_merchants.csv')
# df_test  = pd.read_csv('../data/FEwithoutMerchants/FE_test_without_merchants.csv')

# ---------------------------
# 2) Columns
# ---------------------------
categorical_cols = ['type', 'hourOfDay', 'dayOfWeek', 'transaction_sequence']

# Make categories consistent across splits (optional but good practice)
for col in categorical_cols:
    categories = sorted(pd.concat([df_train[col], df_val[col], df_test[col]], axis=0)
                        .dropna().astype(str).unique())
    # Cast to string before Categorical to avoid mixed types
    df_train[col] = pd.Categorical(df_train[col].astype(str), categories=categories)
    df_val[col]   = pd.Categorical(df_val[col].astype(str),   categories=categories)
    df_test[col]  = pd.Categorical(df_test[col].astype(str),  categories=categories)

selected_features = [
    'step','type','hourOfDay','day','amountLog','dayOfWeek',
    'meanSent','transaction_sequence',
    'totalSent','stdSent','numSent',
    'totalReceived','meanReceived','stdReceived','numReceived','maxAmountReceived',
    'stdAmountReceived','avgAmountToDest','std_to_mean_ratio','pctForwarded24h',
    'transactionRecency','is_early_transaction','sequence_frequency','sequence_count',
    'is_transfer_cashout','is_cashin_transfer','is_cashout_transfer','is_cashin_transfer_cashout',
    'is_transfer_transfer','is_first_transfer','is_cashin_cashout',
    'typeHighValueFlag',
    'pagerank_diff','receiver_indeg_amt','sender_pr','receiver_pr',
    'sender_outdeg_amt','sender_indeg_amt','receiver_outdeg_amt',
    'sender_outdeg_cnt','sender_indeg_cnt','receiver_outdeg_cnt','receiver_indeg_cnt',
    'outdeg_amt_diff','indeg_amt_diff','outdeg_cnt_diff','indeg_cnt_diff'
]

Xtrain, ytrain = df_train[selected_features].copy(), df_train['isFraud'].astype(int).copy()
Xval,   yval   = df_val[selected_features].copy(),   df_val['isFraud'].astype(int).copy()
Xtest,  ytest  = df_test[selected_features].copy(),  df_test['isFraud'].astype(int).copy()

num_cols = [c for c in selected_features if c not in categorical_cols]
cat_cols = categorical_cols

# Replace +/-inf with NaN so imputer can handle it
for dfX in (Xtrain, Xval, Xtest):
    for c in num_cols:
        if c in dfX:
            dfX[c] = pd.to_numeric(dfX[c], errors='coerce').replace([np.inf, -np.inf], np.nan)


In [ ]:

# ---------------------------
# 3) Preprocess + Model
# ---------------------------
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=True, with_std=True)),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    # Use sparse=True for wide compatibility; switch to sparse_output=True if on newer sklearn
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
], remainder="drop", sparse_threshold=1.0)

logreg = LogisticRegression(
    max_iter=1000,
    solver="saga",      # works with sparse matrices
    penalty="l2",
    # class_weight=None  # simplest: no class weights; we'll keep default 0.5 threshold
)

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", logreg),
])

# ---------------------------
# 4) Train on train, quick check on val, then evaluate on test
# ---------------------------
pipe.fit(Xtrain, ytrain)

# Validation (optional quick peek)
proba_val = pipe.predict_proba(Xval)[:, 1]
yhat_val = (proba_val >= 0.50).astype(int)
print("\n=== VALIDATION @ threshold = 0.50 ===")
print("PR-AUC :", average_precision_score(yval, proba_val))
print("ROC-AUC:", roc_auc_score(yval, proba_val))
print(classification_report(yval, yhat_val, digits=4))
print("Confusion matrix:\n", confusion_matrix(yval, yhat_val))

# Test
proba_test = pipe.predict_proba(Xtest)[:, 1]
yhat_test = (proba_test >= 0.50).astype(int)

print("\n=== TEST @ threshold = 0.50 ===")
print("PR-AUC :", average_precision_score(ytest, proba_test))
print("ROC-AUC:", roc_auc_score(ytest, proba_test))
print(classification_report(ytest, yhat_test, digits=4))
print("Confusion matrix:\n", confusion_matrix(ytest, yhat_test))
